In [ ]:
# Innflutningur pakka
from __future__ import annotations

import json
import tarfile
import tempfile
import zipfile
from datetime import datetime
from pathlib import Path
from urllib.request import urlretrieve

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from joblib import parallel_backend
from IPython.display import Image, Markdown, display
from rasterio.features import rasterize
from rasterio.plot import plotting_extent
from rasterio.warp import Resampling, calculate_default_transform, reproject
from scipy.ndimage import distance_transform_edt, uniform_filter
from sklearn.calibration import calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, brier_score_loss, r2_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# ── Skráarslóðir ───────────────────────────────────────────────────────────────
BASE        = Path.cwd()
DATA_DIR    = BASE / 'data'
TERRAIN_DIR = DATA_DIR / 'terrain'
OUTPUT_DIR  = BASE / 'turbine_truth_outputs_report_style'
OUTPUT_DIR.mkdir(exist_ok=True)

DEM_PATH        = TERRAIN_DIR / 'copdem_iceland_3057.tif'
SLOPE_PATH      = TERRAIN_DIR / 'copdem_iceland_slope_3057.tif'
DEM_ARCHIVE     = DATA_DIR / 'rasters_COP90.tar.gz'
DEM_EXTRACT_DIR = TERRAIN_DIR / 'copdem_extract'
UST_ZIP_PATH    = TERRAIN_DIR / 'ust_protected_areas.zip'
UST_SHP_DIR     = TERRAIN_DIR / 'ust_protected_areas'
UST_SHP         = UST_SHP_DIR / 'fridlyst_svaediPolygon.shp'

VINDATLAS_DIR = DATA_DIR / 'vindatlas' / 'rasters'
LINES_PATH    = DATA_DIR / 'Transmission lines.geojson'
TURBINES_PATH = DATA_DIR / 'VindTurbinurHnit.geojson'

# Slóðir að vindatlasgögnum
VINDATLAS_PATHS = {
    'atlas_wind_100m':            VINDATLAS_DIR / 'wind_mean_100m.tif',
    'atlas_power_100m':           VINDATLAS_DIR / 'power_density_100m.tif',
    'atlas_dir_persistence_100m': VINDATLAS_DIR / 'dir_persistence_100m.tif',
    'atlas_weibull_A_100m':       VINDATLAS_DIR / 'weibull_A_100m.tif',
    'atlas_weibull_k_100m':       VINDATLAS_DIR / 'weibull_k_100m.tif',
    'atlas_shear_alpha':          VINDATLAS_DIR / 'shear_alpha.tif',
}

FEATURE_NAMES = list(VINDATLAS_PATHS) + ['slope_deg', 'ruggedness_m', 'dist_line_km', 'elevation_m']

# Ytri gagnaskrár
ICELAND_URL  = 'https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_countries.zip'
GLACIERS_URL = 'https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_glaciated_areas.zip'
URBAN_URL    = 'https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_urban_areas.zip'
AIRPORTS_URL = 'https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_airports.zip'
ROADS_URL    = 'https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_roads.zip'
UST_ZIP_URL  = (
    'https://gis.ust.is/geoserver/fridlyst_svaedi/ows'
    '?maxFeatures=100000&outputFormat=SHAPE-ZIP'
    '&request=GetFeature&service=WFS'
    '&typeName=fridlyst_svaedi:fridlyst_svaedi&version=1.0.0'
)

# ── Líkanabreytur ──────────────────────────────────────────────────────────────
OUTER_FOLDS      = 5
INNER_FOLDS      = 3
RANDOM_SEED      = 42
BG_FRACTION      = 0.5      # hlutfall af handahófs bakgrunnsúrtaki
MIN_DIST_KM      = 1.5      # lágmarks fjarlægð frá þekktum vindmyllum (km)
CELL_KM          = 0.5      # stærð grindureits (km)
BLOCK_KM         = 75       # stærð staðrænna blokka fyrir krossgildingarprófun (km)
THRESHOLD_METRIC = 'f1'
THRESHOLD_GRID   = np.round(np.arange(0.20, 0.82, 0.02), 2)
JOBLIB_BACKEND   = 'threading'
UNSTABLE_Q       = 0.35     # neðri hlutfall til að skilgreina óstöðugt vindsvæði
MODEL_TYPE       = 'RF'     # 'RF' = Random Forest, 'GBT' = Gradient Boosting

# Færibreytuleit fyrir Random Forest
RF_PARAM_GRID = {
    'model__n_estimators':      [300, 500],
    'model__max_depth':         [4, 5, 6],
    'model__min_samples_leaf':  [4, 8, 16],
    'model__min_samples_split': [10, 20],
    'model__max_features':      ['sqrt'],
    'model__class_weight':      ['balanced'],
}

# Færibreytuleit fyrir Gradient Boosting
GBT_PARAM_GRID = {
    'model__learning_rate':     [0.05, 0.1],
    'model__max_depth':         [3, 4],
    'model__min_samples_leaf':  [20, 40],
    'model__max_iter':          [300, 500],
    'model__l2_regularization': [1.0, 5.0],
}

# Yfirlit yfir stillingar
pd.DataFrame([{
    'outer_folds': OUTER_FOLDS, 'inner_folds': INNER_FOLDS,
    'model_type': MODEL_TYPE, 'cell_km': CELL_KM,
    'min_dist_km': MIN_DIST_KM, 'unstable_quantile': UNSTABLE_Q,
    'block_km': BLOCK_KM, 'threshold_metric': THRESHOLD_METRIC,
}])

In [ ]:
# ── Raster og gagnameðhöndlun ──────────────────────────────────────────────────

def read_raster(path):
    """Les raster-skrá og skilar fylki og lýsigögnum."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype('float32')
        meta = src.meta.copy()
        if src.nodata is not None:
            arr = np.where(np.isclose(arr, src.nodata), np.nan, arr)
    return arr, meta


def reproject_to_match(src_path, ref_meta):
    """Umvörpun á raster til að passa við viðmiðunarlýsigögn."""
    with rasterio.open(src_path) as src:
        dst = np.full((ref_meta['height'], ref_meta['width']), np.nan, dtype='float32')
        reproject(
            source=rasterio.band(src, 1), destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref_meta['transform'], dst_crs=ref_meta['crs'],
            resampling=Resampling.bilinear,
        )
        if src.nodata is not None:
            dst = np.where(np.isclose(dst, src.nodata), np.nan, dst)
    return dst.astype('float32')


def local_std(arr, size=11):
    """Reiknar staðræna staðalfrávik (mælikvarði á gróft land)."""
    valid  = np.isfinite(arr)
    filled = np.where(valid, arr, 0.0).astype('float32')
    n      = size * size
    count  = uniform_filter(valid.astype('float32'), size=size, mode='nearest') * n
    s1     = uniform_filter(filled, size=size, mode='nearest') * n
    s2     = uniform_filter(filled ** 2, size=size, mode='nearest') * n
    var    = np.where(count > 0, s2 / count - (s1 / count) ** 2, np.nan)
    return np.sqrt(np.maximum(var, 0.0)).astype('float32')


def sample_at_points(arr, ref_meta, gdf):
    """Sækir raster-gildi á staðsetningum punkta."""
    rows, cols = rasterio.transform.rowcol(ref_meta['transform'], gdf.geometry.x.values, gdf.geometry.y.values)
    out = np.full(len(rows), np.nan, dtype='float32')
    h, w = arr.shape
    for i, (r, c) in enumerate(zip(rows, cols)):
        if 0 <= r < h and 0 <= c < w:
            out[i] = arr[r, c]
    return out


def random_rows_cols(mask, n, rng):
    """Velur handahófskennt n línur/dálka úr Boolean-grímu."""
    rows, cols = np.where(mask)
    idx = rng.choice(len(rows), size=min(n, len(rows)), replace=False)
    return rows[idx], cols[idx]


def _download(url, target):
    """Sækir skrá ef hún er ekki þegar til."""
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(url, target)
    return target


def _download_tmp(url, tmp_dir):
    """Sækir skrá í tímabundinn möppu."""
    target = tmp_dir / Path(url).name
    if not target.exists():
        urlretrieve(url, target)
    return target


def prepare_dem():
    """Útbýr hæðarlíkan (DEM) og hallalag ef þau vantar."""
    TERRAIN_DIR.mkdir(exist_ok=True)
    DEM_EXTRACT_DIR.mkdir(exist_ok=True)
    tifs = sorted(DEM_EXTRACT_DIR.glob('*.tif'))
    if not tifs:
        with tarfile.open(DEM_ARCHIVE, 'r:gz') as tar:
            try:
                tar.extractall(DEM_EXTRACT_DIR, filter='data')
            except TypeError:
                tar.extractall(DEM_EXTRACT_DIR)
        tifs = sorted(DEM_EXTRACT_DIR.glob('*.tif'))

    # Umvörpun á EPSG:3057 (íslenski hnitskerðið)
    if not DEM_PATH.exists():
        with rasterio.open(tifs[0]) as src:
            transform, width, height = calculate_default_transform(
                src.crs, 'EPSG:3057', src.width, src.height, *src.bounds, resolution=90
            )
            meta = src.meta.copy()
            meta.update(crs='EPSG:3057', transform=transform, width=width, height=height, compress='lzw')
            with rasterio.open(DEM_PATH, 'w', **meta) as dst:
                reproject(
                    source=rasterio.band(src, 1), destination=rasterio.band(dst, 1),
                    src_transform=src.transform, src_crs=src.crs,
                    dst_transform=transform, dst_crs='EPSG:3057',
                    resampling=Resampling.bilinear,
                )

    # Reiknar halla úr hæðarlíkaninu
    if not SLOPE_PATH.exists():
        with rasterio.open(DEM_PATH) as src:
            dem = src.read(1).astype('float32')
            if src.nodata is not None:
                dem = np.where(np.isclose(dem, src.nodata), np.nan, dem)
            gy, gx = np.gradient(dem, abs(src.transform.e), src.transform.a)
            slope  = np.degrees(np.arctan(np.sqrt(gx ** 2 + gy ** 2)))
            out    = np.where(np.isfinite(slope), slope, -9999).astype('float32')
            meta   = src.meta.copy()
            meta.update(dtype='float32', nodata=-9999, compress='lzw')
            with rasterio.open(SLOPE_PATH, 'w', **meta) as dst:
                dst.write(out, 1)


def prepare_protected_areas():
    """Sækir og undirbýr friðlýst svæði frá UST."""
    _download(UST_ZIP_URL, UST_ZIP_PATH)
    if not UST_SHP.exists():
        UST_SHP_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(UST_ZIP_PATH) as zf:
            zf.extractall(UST_SHP_DIR)
    return UST_SHP

In [ ]:
# ── Kortagerðarhjálp ──────────────────────────────────────────────────────────

def _draw_overlays(ax, overlays, draw_turbines=True):
    """Teiknar yfirlit yfir land (jöklar, friðlýst svæði, þéttbýli) á ás."""
    for name, color, edge, alpha in [
        ('glaciers',        'lightcyan', 'deepskyblue', 0.25),
        ('protected_areas', '#9ec5ab',  '#3a7d44',     0.20),
        ('urban',           'salmon',   'salmon',       0.15),
    ]:
        gdf = overlays.get(name)
        if gdf is not None and not gdf.empty:
            gdf.plot(ax=ax, color=color, edgecolor=edge, alpha=alpha, zorder=2)

    roads = overlays.get('roads')
    if roads is not None and not roads.empty:
        roads.plot(ax=ax, color='#aaaaaa', linewidth=0.4, alpha=0.6, zorder=3)

    tl = overlays.get('transmission_lines')
    if tl is not None and not tl.empty:
        tl.plot(ax=ax, color='yellow', linewidth=0.7, alpha=0.85, zorder=4)

    if draw_turbines:
        kt = overlays.get('known_turbines')
        if kt is not None and not kt.empty:
            kt.plot(ax=ax, color='white', markersize=8, alpha=0.9,
                    edgecolor='black', linewidth=0.2, zorder=5)


def save_map(path, arr, ref_meta, overlays, background_mask=None,
             cmap='RdYlGn', title='', colorbar_label='Líkindi'):
    """Vistar líkindakort sem myndarskrá."""
    extent = plotting_extent(np.zeros((ref_meta['height'], ref_meta['width'])), ref_meta['transform'])
    fig, ax = plt.subplots(figsize=(13, 10))
    if background_mask is not None:
        ax.imshow(np.ma.masked_invalid(np.where(background_mask, 1.0, np.nan)),
                  extent=extent, origin='upper', cmap='Greys', vmin=0, vmax=2, alpha=0.35)
    im = ax.imshow(np.ma.masked_invalid(arr), extent=extent, origin='upper', cmap=cmap)
    plt.colorbar(im, ax=ax, shrink=0.7, label=colorbar_label)
    _draw_overlays(ax, overlays)
    if title:
        ax.set_title(title, fontsize=13)
    ax.set_axis_off()
    plt.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)

In [ ]:
# ── Skýrsluhjálp ──────────────────────────────────────────────────────────────

def make_table(rows, columns=('Atriði', 'Gildi')):
    """Búar til einfalda DataFrame töflu."""
    return pd.DataFrame(rows, columns=list(columns))


def show_overview(summary):
    """Birtir töfluyfirlit yfir líkan, krossgildingarprófun og kortaútbreiðslu."""
    cv, neg, spread = summary['cv'], summary['negative_sampling'], summary['map_spread']
    cell_km = summary.get('cell_km', 1.0)

    display(Markdown('### Líkan og þjálfunaruppsetning'))
    display(make_table([
        ('Líkan',                         summary['model']),
        ('Upprunalegar vindmyllupunktar',  summary['n_turbine_points']),
        (f'Svæðarreitar ({cell_km:.1f} km grind)', summary['n_site_cells']),
        ('Bakgrunns-neikvæð',             neg['n_background']),
        ('Óstöðugt-vind neikvæð',         neg['n_unstable']),
        ('Þröskuldur stefnuþrálægni',     round(neg['dir_persistence_threshold'], 4)),
        ('Þröskuldur Weibull k',          round(neg['weibull_k_threshold'], 4)),
        ('Bestu færibreytur',             summary['best_params']),
    ]))

    display(Markdown('### Afköst krossgildingarprófunar'))
    display(make_table([
        ('Ytri / innri faldir',   f"{int(cv['outer_folds'])} × {int(cv['inner_folds'])}"),
        ('Staðræn blokkastærð (km)', cv['block_km']),
        ('Valinn þröskuldur',     cv['threshold']),
        ('Meðal ROC-AUC',         round(cv['mean_roc_auc'], 4)),
        ('Meðal meðalþykkni',     round(cv['mean_avg_prec'], 4)),
        ('F1 (samsöfnuð)',        round(cv['f1'], 4)),
        ('Endurköllun (samsöfnuð)', round(cv['recall'], 4)),
        ('Þykkni (samsöfnuð)',    round(cv['precision'], 4)),
        ('Nákvæmni (samsöfnuð)', round(cv['accuracy'], 4)),
    ]))

    display(Markdown('### Kortaþekja'))
    display(make_table([
        ('Spáðir reitir',          spread['predicted_cells']),
        ('Hlutfall af gildri flatarmál', f"{spread['share_of_valid_cells']:.1%}"),
        ('Meðalfjarlægð að vindmyllu', f"{spread['mean_dist_to_turbine_km']:.1f} km"),
        ('Miðgildi fjarlægðar',    f"{spread['median_dist_to_turbine_km']:.1f} km"),
        ('Hlutfall > 10 km frá vindmyllum', f"{spread['share_gt_10km']:.1%}"),
        ('Hlutfall > 20 km frá vindmyllum', f"{spread['share_gt_20km']:.1%}"),
    ]))


def plot_validation_report(summary, threshold_df, oof_df):
    """Teiknar ROC-feril með AUC-gildi."""
    cv = summary['cv']
    fpr, tpr, _ = roc_curve(oof_df['target'], oof_df['predicted_probability'])
    fig, ax = plt.subplots(figsize=(6, 4.8))
    ax.plot(fpr, tpr, color='#4c78a8', linewidth=2)
    ax.plot([0, 1], [0, 1], '--', color='grey', linewidth=1)
    ax.set_title(f"OOF ROC-ferill  (AUC = {cv['mean_roc_auc']:.3f})")
    ax.set_xlabel('Hlutfall rangra jákvæðra')
    ax.set_ylabel('Hlutfall réttra jákvæðra')
    plt.tight_layout()
    plt.show()


def plot_confusion_matrix_report(oof_df, threshold):
    """Sýnir ruglingsmatrix yfir samsöfnuð OOF-spár."""
    y_true = oof_df['target'].values
    y_pred = (oof_df['predicted_probability'].values >= threshold).astype(int)
    cm     = confusion_matrix(y_true, y_pred, labels=[0, 1])

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Spáð Neg', 'Spáð Pos'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['Raunveruleg Neg', 'Raunveruleg Pos'])
    ax.set_title(f'Ruglingsmatrix — samsöfnuð OOF  (þröskuldur = {threshold:.2f})')
    total = cm.sum()
    for i in range(2):
        for j in range(2):
            v = cm[i, j]
            color = 'white' if v > cm.max() * 0.6 else 'black'
            ax.text(j, i, f'{v}\n({v/total:.1%})', ha='center', va='center',
                    color=color, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_threshold_curve(threshold_df, threshold):
    """Sýnir F1, þykkni og endurköllun sem fall af þröskuldinum."""
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(threshold_df['threshold'], threshold_df['f1'],        color='#4c78a8', linewidth=2, label='F1')
    ax.plot(threshold_df['threshold'], threshold_df['precision'], color='#f58518', linewidth=2, label='Þykkni')
    ax.plot(threshold_df['threshold'], threshold_df['recall'],    color='#54a24b', linewidth=2, label='Endurköllun')
    ax.axvline(threshold, color='crimson', linestyle='--', linewidth=1.5,
               label=f'Valinn þröskuldur ({threshold:.2f})')
    ax.set_xlabel('Þröskuldur'); ax.set_ylabel('Skor')
    ax.set_title('F1, þykkni og endurköllun sem fall af þröskuldinum')
    ax.set_xlim(THRESHOLD_GRID[0], THRESHOLD_GRID[-1]); ax.set_ylim(0, 1.05)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_feature_importance(imp_df):
    """Birtir töflu og súlurit yfir mikilvægi breyta."""
    display(imp_df.round(4))
    plot_df = imp_df.sort_values('importance', ascending=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(plot_df['feature'], plot_df['importance'], color='#4c78a8')
    ax.set_title('Mikilvægi breyta')
    ax.set_xlabel('Mikilvægi')
    plt.tight_layout()
    plt.show()


def plot_feature_distributions(imp_df, positives, negatives):
    """Ber saman dreifingu efstu 4 breyta á milli jákvæðra og neikvæðra dæma."""
    top4 = imp_df.head(4)['feature'].tolist()
    all_samples = pd.concat([positives, negatives], ignore_index=True)
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    for ax, feat in zip(axes.flatten(), top4):
        ax.hist(all_samples.loc[all_samples['target'] == 0, feat], bins=25, alpha=0.6,
                density=True, color='#4c78a8', label='Neikvæð')
        ax.hist(all_samples.loc[all_samples['target'] == 1, feat], bins=25, alpha=0.6,
                density=True, color='#f58518', label='Jákvæð')
        ax.set_title(feat, fontsize=10)
        ax.tick_params(axis='x', rotation=20, labelsize=8)
    axes.flatten()[0].legend(fontsize=9)
    fig.suptitle('Dreifing breyta — jákvæð vs. neikvæð dæmi', fontsize=11)
    plt.tight_layout()
    plt.show()


def show_fold_report(fold_df):
    """Birtir niðurstöður hvers faldur og súlurit yfir helstu mælikvarða."""
    cols = ['fold', 'roc_auc', 'avg_prec', 'f1', 'precision', 'recall', 'accuracy', 'best_inner_recall']
    display(Markdown('### Niðurstöður hvers faldur'))
    display(fold_df[cols].round(4))
    x = np.arange(len(fold_df)); width = 0.25
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x - width, fold_df['roc_auc'], width=width, label='ROC-AUC', color='#4c78a8')
    ax.bar(x,         fold_df['f1'],      width=width, label='F1',      color='#f58518')
    ax.bar(x + width, fold_df['recall'],  width=width, label='Endurköllun', color='#54a24b')
    ax.set_xticks(x); ax.set_xticklabels([f'Faldur {i}' for i in fold_df['fold']])
    ax.set_ylim(0, 1.05); ax.set_title('Mælikvarðar hvers faldur'); ax.legend()
    plt.tight_layout()
    plt.show()


def plot_uncertainty_report(unc_map, ref_meta, overlays, valid_mask):
    """Sýnir óvissukortsyfirlit yfir Ísland."""
    extent = plotting_extent(np.zeros((ref_meta['height'], ref_meta['width'])), ref_meta['transform'])
    fig, ax = plt.subplots(figsize=(13, 10))
    if valid_mask is not None:
        ax.imshow(np.ma.masked_invalid(np.where(valid_mask, 1.0, np.nan)),
                  extent=extent, origin='upper', cmap='Greys', vmin=0, vmax=2, alpha=0.35)
    im = ax.imshow(np.ma.masked_invalid(unc_map), extent=extent, origin='upper', cmap='plasma_r')
    plt.colorbar(im, ax=ax, shrink=0.7, label='Óvissa (std yfir tré)')
    _draw_overlays(ax, overlays)
    ax.set_title('Óvissukor spálíkans', fontsize=13)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Hleðsla á staðrænni gögnum ─────────────────────────────────────────────────

def load_layers_and_context():
    """Hleður öllum raster-lögum, landmælingum og þekktum vindmyllum."""
    prepare_dem()
    dem, ref_meta = read_raster(DEM_PATH)
    slope, _      = read_raster(SLOPE_PATH)

    # Hlaða vindatlasgögnum og endurvarpa ef þörf krefur
    layers = {}
    for name, path in VINDATLAS_PATHS.items():
        arr, meta = read_raster(path)
        if meta['crs'] != ref_meta['crs'] or meta['height'] != ref_meta['height']:
            arr = reproject_to_match(path, ref_meta)
        layers[name] = arr

    # Sækja kortgögn úr Natural Earth
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        countries = gpd.read_file(_download_tmp(ICELAND_URL, tmp))
        iceland   = countries[countries['ADMIN'] == 'Iceland'].to_crs(4326)
        glaciers  = gpd.clip(gpd.read_file(_download_tmp(GLACIERS_URL, tmp)).to_crs(4326), iceland)
        urban     = gpd.clip(gpd.read_file(_download_tmp(URBAN_URL,    tmp)).to_crs(4326), iceland)
        airports  = gpd.clip(gpd.read_file(_download_tmp(AIRPORTS_URL, tmp)).to_crs(4326), iceland)
        roads_ne  = gpd.clip(gpd.read_file(_download_tmp(ROADS_URL,    tmp)).to_crs(4326), iceland)

    crs = ref_meta['crs']
    proj = {k: v.to_crs(crs) for k, v in {
        'iceland':   iceland,
        'glaciers':  glaciers,
        'urban':     urban,
        'airports':  airports,
        'protected': gpd.read_file(prepare_protected_areas()).to_crs(4326),
    }.items()}

    lines    = gpd.read_file(LINES_PATH).to_crs(crs)
    turbines = gpd.read_file(TURBINES_PATH)
    if turbines.crs is None:
        turbines = turbines.set_crs(4326)
    turbines = turbines.to_crs(crs)

    shape, T = (ref_meta['height'], ref_meta['width']), ref_meta['transform']

    def burn(gdf):
        """Umbreytir GeoDataFrame í Boolean raster."""
        if gdf.empty:
            return np.zeros(shape, dtype=bool)
        return rasterize([(g, 1) for g in gdf.geometry if g],
                         out_shape=shape, transform=T, fill=0, dtype='uint8').astype(bool)

    # Búa til grímur fyrir land og útilokunarsvæði
    land_mask = burn(proj['iceland'])
    excl_mask = burn(proj['glaciers']) | burn(proj['protected']) | burn(proj['urban']) | burn(proj['airports'])
    line_rast = rasterize([(g, 1) for g in lines.geometry if g],
                          out_shape=shape, transform=T, fill=0, dtype='uint8')

    # Reikna afleiddar breytur
    layers['slope_deg']    = slope
    layers['ruggedness_m'] = local_std(dem, size=11)
    layers['dist_line_km'] = (distance_transform_edt(1 - line_rast) * abs(T.a) / 1000.0).astype('float32')
    layers['elevation_m']  = dem

    masks = {'land': land_mask, 'exclusion': excl_mask}
    overlays = {
        'glaciers':           proj['glaciers'],
        'protected_areas':    proj['protected'],
        'urban':              proj['urban'],
        'airports':           proj['airports'],
        'known_turbines':     turbines,
        'transmission_lines': lines,
        'roads':              roads_ne.to_crs(crs),
    }
    return layers, ref_meta, overlays, masks, turbines


# ── Þjálfunargögn ─────────────────────────────────────────────────────────────

def aggregate_positives(turbines, layers, ref_meta, cell_km=1.0):
    """Sameinar vindmyllustaðsetningar í grindureiti og sækir breytur."""
    m   = cell_km * 1000.0
    df  = pd.DataFrame({'x': turbines.geometry.x, 'y': turbines.geometry.y})
    df['gx'] = np.floor(df['x'] / m).astype(int)
    df['gy'] = np.floor(df['y'] / m).astype(int)
    grp = df.groupby(['gx', 'gy'], as_index=False).agg(
        x=('x', 'mean'), y=('y', 'mean'), n_turbines=('x', 'size')
    )
    gdf = gpd.GeoDataFrame(grp, geometry=gpd.points_from_xy(grp['x'], grp['y']), crs=ref_meta['crs'])
    for name in FEATURE_NAMES:
        grp[name] = sample_at_points(layers[name], ref_meta, gdf)
    grp['sample_type'] = 'known_site_cell'
    grp['target']      = 1
    grp.insert(0, 'sample_id', np.arange(len(grp)))
    return grp.drop(columns=['gx', 'gy']).dropna().reset_index(drop=True)


def build_mixed_negatives(layers, masks, turbines, ref_meta, n_points):
    """Smíðar blönduð neikvæð dæmi úr bakgrunni og óstöðugum vindsvæðum."""
    shape, T = (ref_meta['height'], ref_meta['width']), ref_meta['transform']

    # Útilokar svæði nálægt þekktum vindmyllum
    turbine_buf = rasterize(
        [(g.buffer(MIN_DIST_KM * 1000), 1) for g in turbines.geometry if g],
        out_shape=shape, transform=T, fill=0, dtype='uint8',
    ).astype(bool)

    candidate = masks['land'] & ~masks['exclusion'] & ~turbine_buf
    for name in FEATURE_NAMES:
        candidate &= np.isfinite(layers[name])

    # Skilgreinir óstöðug vindsvæði með neðri hlutfallsgildum
    dir_thr = float(np.quantile(layers['atlas_dir_persistence_100m'][candidate], UNSTABLE_Q))
    k_thr   = float(np.quantile(layers['atlas_weibull_k_100m'][candidate], UNSTABLE_Q))
    unstable = (
        candidate
        & (layers['atlas_dir_persistence_100m'] <= dir_thr)
        & (layers['atlas_weibull_k_100m']        <= k_thr)
    )

    n_bg  = int(round(n_points * BG_FRACTION))
    n_ust = min(n_points - n_bg, int(unstable.sum()))
    n_bg  = n_points - n_ust

    parts = []
    for mask, label, seed, n in [
        (candidate, 'background_random',        RANDOM_SEED,     n_bg),
        (unstable,  'background_unstable_wind', RANDOM_SEED + 1, n_ust),
    ]:
        rows, cols = random_rows_cols(mask, n, np.random.default_rng(seed))
        xs, ys     = rasterio.transform.xy(T, rows, cols, offset='center')
        df = pd.DataFrame({
            'x': np.asarray(xs, 'float64'), 'y': np.asarray(ys, 'float64'),
            'n_turbines': 0, 'sample_type': label, 'target': 0,
        })
        for name in FEATURE_NAMES:
            df[name] = layers[name][rows, cols]
        parts.append(df.dropna())

    out = pd.concat(parts, ignore_index=True).reset_index(drop=True)
    out.insert(0, 'sample_id', np.arange(len(out)))
    neg_meta = {
        'dir_persistence_threshold': dir_thr,
        'weibull_k_threshold':       k_thr,
        'n_background':              n_bg,
        'n_unstable':                n_ust,
    }
    return out, neg_meta


def assign_blocks(df):
    """Úthlutar staðrænum blokkum til hverrar sýnar."""
    m   = BLOCK_KM * 1000.0
    out = df.copy()
    out['block_id'] = (
        np.floor(out['x'] / m).astype(int).astype(str) + '_' +
        np.floor(out['y'] / m).astype(int).astype(str)
    )
    return out


# ── Líkana-hjálp ──────────────────────────────────────────────────────────────

def _model_pipeline():
    """Skilar sklearn-pipeline með meðhöndlun á nullgildum og flokkara."""
    if MODEL_TYPE == 'GBT':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model',   HistGradientBoostingClassifier(random_state=RANDOM_SEED, class_weight='balanced')),
        ])
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model',   RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1)),
    ])


def _active_param_grid():
    return GBT_PARAM_GRID if MODEL_TYPE == 'GBT' else RF_PARAM_GRID


def _pos_proba(model, X):
    """Skilar líkindum fyrir flokkinn 1 (jákvæður)."""
    proba   = model.predict_proba(X)
    classes = list(model.named_steps['model'].classes_)
    if 1 in classes:
        return proba[:, classes.index(1)]
    return np.zeros(len(X), dtype='float32')


def _metric_row(label, y_true, scores, threshold):
    """Reiknar helstu mælikvarða fyrir eitt þröskuldgildi."""
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        'label': label, 'threshold': threshold,
        'roc_auc':   float(roc_auc_score(y_true, scores)),
        'avg_prec':  float(average_precision_score(y_true, scores)),
        'f1':        float(f1_score(y_true, pred, zero_division=0)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall':    float(recall_score(y_true, pred, zero_division=0)),
        'accuracy':  float(accuracy_score(y_true, pred)),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


def _tune_threshold(y_true, scores):
    """Finnur besta þröskuldgildi sem hámarkar valinn mælikvarða."""
    df   = pd.DataFrame([_metric_row(f't={t:.2f}', y_true, scores, float(t)) for t in THRESHOLD_GRID])
    best = df.sort_values([THRESHOLD_METRIC, 'recall', 'threshold'], ascending=[False, False, True]).iloc[0]
    return df, float(best['threshold'])


# ── Krossgildingarprófun og líkanapassun ───────────────────────────────────────

def run_nested_cv(training_df):
    """Keyrir hreiðraða staðræna krossgildingarprófun og skilar niðurstöðum."""
    X, y, groups = training_df[FEATURE_NAMES], training_df['target'].astype(int), training_df['block_id']
    outer_cv = StratifiedGroupKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_SEED)

    fold_rows, oof_parts = [], []
    for fold, (tr, te) in enumerate(outer_cv.split(X, y, groups), start=1):
        inner_cv = StratifiedGroupKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_SEED + fold)
        search   = GridSearchCV(_model_pipeline(), _active_param_grid(), scoring='recall',
                                cv=inner_cv, n_jobs=-1, refit=True)
        with parallel_backend(JOBLIB_BACKEND):
            search.fit(X.iloc[tr], y.iloc[tr], groups=groups.iloc[tr])

        scores = _pos_proba(search.best_estimator_, X.iloc[te])
        row    = _metric_row(f'fold_{fold}', y.iloc[te], scores, 0.5)
        row.update({'fold': fold, 'best_inner_recall': search.best_score_,
                    'best_params': json.dumps(search.best_params_, sort_keys=True)})
        fold_rows.append(row)

        oof = training_df.iloc[te][['sample_id', 'sample_type', 'target', 'x', 'y', 'block_id']].copy()
        oof['fold'] = fold
        oof['predicted_probability'] = scores
        oof_parts.append(oof)

    fold_df = pd.DataFrame(fold_rows)
    oof_df  = pd.concat(oof_parts, ignore_index=True).sort_values('sample_id').reset_index(drop=True)
    return fold_df, oof_df


def fit_final_model(training_df):
    """Þjálfar lokalíkanið á öllum þjálfunargögnum."""
    inner_cv = StratifiedGroupKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    search   = GridSearchCV(_model_pipeline(), _active_param_grid(), scoring='recall',
                            cv=inner_cv, n_jobs=-1, refit=True)
    with parallel_backend(JOBLIB_BACKEND):
        search.fit(training_df[FEATURE_NAMES], training_df['target'], groups=training_df['block_id'])
    params = {k.replace('model__', ''): v for k, v in search.best_params_.items()}
    return search.best_estimator_, params


def _feature_importances(model, training_df):
    """Skilar mikilvægi breyta (innra eða umraðunar-mikilvægi)."""
    if MODEL_TYPE == 'RF':
        return model.named_steps['model'].feature_importances_
    result = permutation_importance(
        model, training_df[FEATURE_NAMES], training_df['target'],
        n_repeats=5, random_state=RANDOM_SEED, n_jobs=-1,
    )
    return result.importances_mean


def predict_probability_map(model, layers, masks, ref_meta):
    """Spáir líkindagildum yfir allt gildanlegt landsvæði á Íslandi."""
    valid = masks['land'] & ~masks['exclusion']
    for name in FEATURE_NAMES:
        valid &= np.isfinite(layers[name])
    rows, cols = np.where(valid)
    prob_map   = np.full((ref_meta['height'], ref_meta['width']), np.nan, dtype='float32')
    # Vinnur í hlutum til að spara minni
    for s in range(0, len(rows), 200_000):
        sl    = slice(s, s + 200_000)
        chunk = pd.DataFrame({n: layers[n][rows[sl], cols[sl]] for n in FEATURE_NAMES})
        prob_map[rows[sl], cols[sl]] = _pos_proba(model, chunk).astype('float32')
    return prob_map, valid


def compute_uncertainty_map(model, layers, masks, ref_meta):
    """Reiknar óvissu líkansins (staðalfrávik yfir tré í RF)."""
    valid = masks['land'] & ~masks['exclusion']
    for name in FEATURE_NAMES:
        valid &= np.isfinite(layers[name])
    rows, cols = np.where(valid)
    unc_map    = np.full((ref_meta['height'], ref_meta['width']), np.nan, dtype='float32')

    if MODEL_TYPE != 'RF':
        # Nálgun fyrir GBT: 2 * min(p, 1-p)
        for s in range(0, len(rows), 200_000):
            sl    = slice(s, s + 200_000)
            chunk = pd.DataFrame({n: layers[n][rows[sl], cols[sl]] for n in FEATURE_NAMES})
            p     = _pos_proba(model, chunk).astype('float32')
            unc_map[rows[sl], cols[sl]] = 2 * np.minimum(p, 1 - p)
        return unc_map

    # RF: staðalfrávik yfir spár allra trjánna
    imputer = model.named_steps['imputer']
    rf      = model.named_steps['model']
    classes = list(rf.classes_)
    pos_idx = classes.index(1) if 1 in classes else None

    for s in range(0, len(rows), 200_000):
        sl    = slice(s, s + 200_000)
        chunk = pd.DataFrame({n: layers[n][rows[sl], cols[sl]] for n in FEATURE_NAMES})
        X_imp = imputer.transform(chunk)
        if pos_idx is None:
            unc_map[rows[sl], cols[sl]] = 0.0
            continue
        tree_preds = np.empty((len(rf.estimators_), X_imp.shape[0]), dtype='float32')
        for i, tree in enumerate(rf.estimators_):
            tree_preds[i] = tree.predict_proba(X_imp)[:, pos_idx]
        unc_map[rows[sl], cols[sl]] = tree_preds.std(axis=0)

    return unc_map

In [ ]:
# ── Aðalferlill ───────────────────────────────────────────────────────────────

def run_pipeline():
    """Keyrir allan ferlilinn frá gagnalæsingu að kortagerð og skýrslum."""

    print('Hleður lögum...')
    layers, ref_meta, overlays, masks, turbines = load_layers_and_context()

    print('Smíðar þjálfunargögn...')
    positives = aggregate_positives(turbines, layers, ref_meta, cell_km=CELL_KM)
    negatives, neg_meta = build_mixed_negatives(layers, masks, turbines, ref_meta,
                                                n_points=len(positives) * 6)
    training_df = assign_blocks(
        pd.concat([positives, negatives], ignore_index=True)
        .dropna(subset=FEATURE_NAMES + ['x', 'y', 'target'])
        .reset_index(drop=True)
    )
    training_df['sample_id'] = np.arange(len(training_df))
    n_pos = int((training_df['target'] == 1).sum())
    n_neg = int((training_df['target'] == 0).sum())
    print(f'  {n_pos} jákvæðir reitir, {n_neg} neikvæð, '
          f'{training_df["block_id"].nunique()} staðrænnar blokkur')

    print('Keyrir hreiðraða staðræna krossgildingarprófun...')
    fold_df, oof_df = run_nested_cv(training_df)

    # Kvarðar líkindaspár með isotonic regression
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(oof_df['predicted_probability'].values, oof_df['target'].values)
    cal_oof = calibrator.predict(oof_df['predicted_probability'].values)
    oof_df  = oof_df.copy()
    oof_df['predicted_probability'] = cal_oof

    threshold_df, threshold = _tune_threshold(oof_df['target'], cal_oof)
    pooled = _metric_row('pooled_oof', oof_df['target'], cal_oof, threshold)
    cv_summary = pd.DataFrame([{
        'outer_folds': OUTER_FOLDS, 'inner_folds': INNER_FOLDS,
        'block_km': BLOCK_KM, 'threshold': threshold,
        'mean_roc_auc':  round(fold_df['roc_auc'].mean(), 4),
        'mean_avg_prec': round(fold_df['avg_prec'].mean(), 4),
        **{k: pooled[k] for k in ['precision', 'recall', 'f1', 'accuracy', 'tn', 'fp', 'fn', 'tp']},
    }])
    print(f'  Þröskuldur: {threshold}  |  F1: {cv_summary.loc[0,"f1"]:.3f}  |  AUC: {cv_summary.loc[0,"mean_roc_auc"]:.3f}')

    print('Þjálfar lokalíkan...')
    final_model, best_params = fit_final_model(training_df)
    importances = _feature_importances(final_model, training_df)
    imp_df = (
        pd.DataFrame({'feature': FEATURE_NAMES, 'importance': importances})
        .sort_values('importance', ascending=False)
        .reset_index(drop=True)
    )

    print('Smíðar líkindakort...')
    prob_map_raw, valid_mask = predict_probability_map(final_model, layers, masks, ref_meta)
    prob_map = np.full_like(prob_map_raw, np.nan)
    finite   = np.isfinite(prob_map_raw)
    prob_map[finite] = calibrator.predict(prob_map_raw[finite]).astype('float32')
    thresh_map = np.where(prob_map >= threshold, prob_map, np.nan).astype('float32')

    print('Reiknar óvissukortsyfirlit (RF: nokkrar mínútur)...')
    uncertainty_map = compute_uncertainty_map(final_model, layers, masks, ref_meta)

    pred_mask  = np.isfinite(thresh_map)
    point_mask = rasterize(
        [(g, 1) for g in turbines.geometry if g],
        out_shape=(ref_meta['height'], ref_meta['width']),
        transform=ref_meta['transform'], fill=0, dtype='uint8',
    ).astype(bool)
    dist_km = distance_transform_edt(~point_mask) * abs(ref_meta['transform'].a) / 1000.0

    print('Vistar niðurstöður...')
    save_map(OUTPUT_DIR / 'thresholded_map.png', thresh_map, ref_meta, overlays,
             background_mask=valid_mask,
             title=f'Hæfni vindmyllna — Þröskuldur {threshold:.2f}')
    save_map(OUTPUT_DIR / 'uncertainty_map.png', uncertainty_map, ref_meta, overlays,
             background_mask=valid_mask, cmap='plasma_r',
             title='Óvissukor spálíkans',
             colorbar_label='Óvissa (std yfir tré)')

    # Vista CSV-niðurstöður
    fold_df.to_csv(     OUTPUT_DIR / 'cv_fold_results.csv',    index=False)
    cv_summary.to_csv(  OUTPUT_DIR / 'cv_summary.csv',         index=False)
    oof_df.to_csv(      OUTPUT_DIR / 'oof_predictions.csv',    index=False)
    threshold_df.to_csv(OUTPUT_DIR / 'threshold_tuning.csv',   index=False)
    imp_df.to_csv(      OUTPUT_DIR / 'feature_importance.csv', index=False)

    cm_row = threshold_df.loc[np.isclose(threshold_df['threshold'], threshold)].iloc[0]
    pd.DataFrame(
        [[int(cm_row['tn']), int(cm_row['fp'])],
         [int(cm_row['fn']), int(cm_row['tp'])]],
        index=['actual_neg', 'actual_pos'],
        columns=['pred_neg', 'pred_pos'],
    ).to_csv(OUTPUT_DIR / 'confusion_matrix.csv')

    # Búa til JSON-samantekt
    model_name = 'HistGradientBoostingClassifier' if MODEL_TYPE == 'GBT' else 'RandomForestClassifier'
    summary = {
        'model':             model_name,
        'best_params':       best_params,
        'n_turbine_points':  int(len(turbines)),
        'n_site_cells':      int(len(positives)),
        'cell_km':           float(CELL_KM),
        'n_negatives':       int(len(negatives)),
        'negative_sampling': neg_meta,
        'features':          FEATURE_NAMES,
        'map_threshold':     threshold,
        'cv':                cv_summary.iloc[0].to_dict(),
        'map_spread': {
            'predicted_cells':           int(pred_mask.sum()),
            'share_of_valid_cells':      float(pred_mask.sum() / valid_mask.sum()),
            'mean_dist_to_turbine_km':   float(np.nanmean(dist_km[pred_mask])),
            'median_dist_to_turbine_km': float(np.nanmedian(dist_km[pred_mask])),
            'share_gt_10km':             float(np.mean(dist_km[pred_mask] > 10.0)),
            'share_gt_20km':             float(np.mean(dist_km[pred_mask] > 20.0)),
        },
    }
    (OUTPUT_DIR / 'summary.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print('Lokið.')

    return (
        summary, fold_df, cv_summary, oof_df, threshold_df,
        imp_df, prob_map, thresh_map, valid_mask, ref_meta, overlays, turbines,
        positives, negatives, uncertainty_map,
    )

In [ ]:
(
    summary, fold_df, cv_summary, oof_df, threshold_df,
    imp_df, prob_map, thresh_map, valid_mask, ref_meta, overlays, turbines,
    positives, negatives, uncertainty_map,
) = run_pipeline()

In [ ]:
# Kortayfirlit yfir rannsóknarsvæðið
display(Markdown('### Rannsóknarsvæðið — útilokunarlag og innviðir'))

_extent = plotting_extent(np.zeros((ref_meta['height'], ref_meta['width'])), ref_meta['transform'])
fig, ax = plt.subplots(figsize=(13, 10))

ax.imshow(np.ma.masked_invalid(np.where(valid_mask, 1.0, np.nan)),
          extent=_extent, origin='upper', cmap='Greys', vmin=0, vmax=2, alpha=0.2)

_legend_handles = []
for _name, _color, _edge, _alpha, _label in [
    ('glaciers',        'lightcyan', 'deepskyblue', 0.6, 'Jöklar'),
    ('protected_areas', '#9ec5ab',   '#3a7d44',     0.5, 'Friðlýst svæði'),
    ('urban',           'salmon',    'firebrick',   0.5, 'Þéttbýli'),
]:
    _gdf = overlays.get(_name)
    if _gdf is not None and not _gdf.empty:
        _gdf.plot(ax=ax, color=_color, edgecolor=_edge, alpha=_alpha, zorder=2)
        _legend_handles.append(mpatches.Patch(facecolor=_color, edgecolor=_edge, label=_label))

_airports = overlays.get('airports')
if _airports is not None and not _airports.empty:
    _airports.plot(ax=ax, color='#e377c2', marker='s', markersize=6, alpha=0.9, zorder=3)
    _legend_handles.append(plt.Line2D([0], [0], marker='s', color='w',
                                       markerfacecolor='#e377c2', markersize=7, label='Flugvellir'))

_roads = overlays.get('roads')
if _roads is not None and not _roads.empty:
    _roads.plot(ax=ax, color='#888888', linewidth=0.5, alpha=0.7, zorder=4)
    _legend_handles.append(plt.Line2D([0], [0], color='#888888', linewidth=1.5, label='Vegir'))

_tl = overlays.get('transmission_lines')
if _tl is not None and not _tl.empty:
    _tl.plot(ax=ax, color='gold', linewidth=1.0, alpha=0.9, zorder=5)
    _legend_handles.append(plt.Line2D([0], [0], color='gold', linewidth=1.5, label='Rafmagnsstrengir'))

_kt = overlays.get('known_turbines')
if _kt is not None and not _kt.empty:
    _kt.plot(ax=ax, color='white', markersize=5, alpha=0.9, edgecolor='black', linewidth=0.3, zorder=6)
    _legend_handles.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='white',
                                       markeredgecolor='black', markersize=6, label='Þekktar vindmyllur'))

ax.legend(handles=_legend_handles, loc='lower left', fontsize=9, framealpha=0.85)
ax.set_title('Rannsóknarsvæðið — útilokunarlag og innviðir', fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Yfirlit yfir líkan og krossgildingarprófun
show_overview(summary)

In [ ]:
# ROC-ferill
plot_validation_report(summary, threshold_df, oof_df)

In [ ]:
# Ruglingsmatrix
plot_confusion_matrix_report(oof_df, summary['map_threshold'])

In [ ]:
# Þröskuldur-ferill
plot_threshold_curve(threshold_df, summary['map_threshold'])

In [ ]:
# Mikilvægi breyta
plot_feature_importance(imp_df)

In [ ]:
# Dreifing breyta: jákvæð vs. neikvæð
plot_feature_distributions(imp_df, positives, negatives)

In [ ]:
# Niðurstöður hvers faldur
show_fold_report(fold_df)

In [ ]:
# Óvissukortsyfirlit
plot_uncertainty_report(uncertainty_map, ref_meta, overlays, valid_mask)

In [ ]:
# Lokakort með þröskuldgildi
display(Markdown(f'### Þröskuldat hæfnikort  (líkindi ≥ {summary["map_threshold"]:.2f})'))
display(Image(filename=str(OUTPUT_DIR / 'thresholded_map.png')))

In [ ]:
# Kvarðunarmælikvarðar líkindaspáa
r2    = r2_score(oof_df['target'], oof_df['predicted_probability'])
brier = brier_score_loss(oof_df['target'], oof_df['predicted_probability'])

display(Markdown('### Kvarðunarmælikvarðar líkindaspáa (samsöfnuð OOF)'))
display(pd.DataFrame([{
    'Líkan':        'RandomForestClassifier',
    'R²':           round(r2, 4),
    'Brier Score':  round(brier, 4),
    'ROC-AUC':      round(summary['cv']['mean_roc_auc'], 4),
    'F1':           round(float(summary['cv']['f1']), 4),
}]))

In [ ]:
# Vista JSON-skýrslu
def _safe(v):
    """Skilar None í stað NaN til að gæta JSON-gildis."""
    return None if isinstance(v, float) and np.isnan(v) else v

_threshold = summary['map_threshold']
_y_true = oof_df['target'].values
_y_pred = (oof_df['predicted_probability'].values >= _threshold).astype(int)
_tn, _fp, _fn, _tp = confusion_matrix(_y_true, _y_pred, labels=[0, 1]).ravel()
cv = summary['cv']

report = {
    'title':        'Hæfnilíkan vindmyllna — Ísland',
    'generated_at': datetime.now().isoformat(timespec='seconds'),
    'settings': {
        'model_type':        MODEL_TYPE,
        'outer_folds':       OUTER_FOLDS,
        'inner_folds':       INNER_FOLDS,
        'block_km':          BLOCK_KM,
        'bg_fraction':       BG_FRACTION,
        'unstable_quantile': UNSTABLE_Q,
        'min_dist_km':       MIN_DIST_KM,
        'threshold_metric':  THRESHOLD_METRIC,
        'n_features':        len(FEATURE_NAMES),
        'features':          FEATURE_NAMES,
    },
    'training_data': {
        'n_turbine_points':  summary['n_turbine_points'],
        'n_site_cells':      summary['n_site_cells'],
        'n_negatives':       summary['n_negatives'],
        'negative_sampling': {
            'n_background':              summary['negative_sampling']['n_background'],
            'n_unstable_wind':           summary['negative_sampling']['n_unstable'],
            'dir_persistence_threshold': round(summary['negative_sampling']['dir_persistence_threshold'], 4),
            'weibull_k_threshold':       round(summary['negative_sampling']['weibull_k_threshold'], 4),
        },
    },
    'model': {
        'type':        summary['model'],
        'best_params': summary['best_params'],
        'calibration': 'isotonic regression þjálfuð á OOF-spám',
    },
    'cross_validation': {
        'selected_threshold': _threshold,
        'mean_roc_auc':       _safe(round(cv['mean_roc_auc'],  4)),
        'mean_avg_precision': _safe(round(cv['mean_avg_prec'], 4)),
        'pooled_f1':          round(float(cv['f1']),        4),
        'pooled_precision':   round(float(cv['precision']),  4),
        'pooled_recall':      round(float(cv['recall']),     4),
        'pooled_accuracy':    round(float(cv['accuracy']),   4),
        'confusion_matrix': {
            'true_negatives':  int(_tn),
            'false_positives': int(_fp),
            'false_negatives': int(_fn),
            'true_positives':  int(_tp),
        },
        'per_fold': [
            {
                'fold':             int(row['fold']),
                'roc_auc':          _safe(round(float(row['roc_auc']),  4)),
                'avg_precision':    _safe(round(float(row['avg_prec']), 4)),
                'f1':               round(float(row['f1']),        4),
                'precision':        round(float(row['precision']),  4),
                'recall':           round(float(row['recall']),     4),
                'accuracy':         round(float(row['accuracy']),   4),
                'n_test_positives': int(row['tp'] + row['fn']),
                'n_test_negatives': int(row['tn'] + row['fp']),
            }
            for _, row in fold_df.iterrows()
        ],
    },
    'feature_importance': [
        {'rank': int(i + 1), 'feature': row['feature'], 'importance': round(float(row['importance']), 6)}
        for i, (_, row) in enumerate(imp_df.iterrows())
    ],
    'map_coverage': {
        'threshold':                  _threshold,
        'predicted_cells':            summary['map_spread']['predicted_cells'],
        'share_of_valid_iceland_pct': round(summary['map_spread']['share_of_valid_cells'] * 100, 2),
        'mean_dist_to_turbine_km':    round(summary['map_spread']['mean_dist_to_turbine_km'],   2),
        'median_dist_to_turbine_km':  round(summary['map_spread']['median_dist_to_turbine_km'], 2),
        'share_gt_10km_pct':          round(summary['map_spread']['share_gt_10km'] * 100, 2),
        'share_gt_20km_pct':          round(summary['map_spread']['share_gt_20km'] * 100, 2),
    },
}

report_path = OUTPUT_DIR / 'report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Vistað → {report_path}')
display(report)